# 1. Escolha do Município e Obtenção dos Dados:

In [1]:
import folium
import pandas as pd
import numpy as np

# Leitura dos DataFrames de Ouro Branco (Votação das Sessões Eleitorais e Local de Votação dos Eleitorados):
df_voting_session = pd.read_csv('votacao_secao_2026_MG_Ouro_Branco.csv', sep=';', encoding='latin_1')
df_electorate_voting_location = pd.read_csv('eleitorado_local_votacao_2026_MG_Ouro_Branco.csv', sep=';', encoding='latin_1')

# 2. Limpeza, Filtragem e Unificação de Dados:

In [5]:
# Adquirindo a localização das sessões eleitorais:

# Padronizando as chaves geográficas, selecionando as colunas para obter a localização de cada seção nas zonas eleitorais:
df_location = df_electorate_voting_location[
    ['NR_TURNO', 'NR_ZONA', 'NR_SECAO', 'NR_LATITUDE', 'NR_LONGITUDE']
    ].drop_duplicates(subset=['NR_TURNO', 'NR_ZONA', 'NR_SECAO'])

# Convertendo as coordenadas para valores numéricos:
df_location['NR_LATITUDE'] = df_location['NR_LATITUDE'].astype(str).str.replace(',', '.')
df_location['NR_LONGITUDE'] = df_location['NR_LONGITUDE'].astype(str).str.replace(',', '.')

df_location['NR_LATITUDE'] = pd.to_numeric(df_location['NR_LATITUDE'], errors='coerce')
df_location['NR_LONGITUDE'] = pd.to_numeric(df_location['NR_LONGITUDE'], errors='coerce')

# Removendo coordenadas nulas ou com o valor -1:
df_location = df_location.dropna(subset=['NR_LATITUDE', 'NR_LONGITUDE'])
df_location = df_location[(df_location['NR_LATITUDE'] != -1) & (df_location['NR_LONGITUDE'] != -1)]

# DataFrame com todos os dados que serão utilizados:
df = df_voting_session[['NR_TURNO', 'CD_ELEICAO', 'NR_ZONA', 'NR_SECAO', 'CD_CARGO', 
                        'DS_CARGO', 'NR_VOTAVEL', 'NM_VOTAVEL', 'QT_VOTOS', 'NM_LOCAL_VOTACAO', 'DS_LOCAL_VOTACAO_ENDERECO']].copy()

# Fazendo o merge das colunas 'NR_LATITUDE' e 'NR_LONGITUDE' no DataFrame:
df = df.merge(df_location, on=['NR_TURNO', 'NR_ZONA', 'NR_SECAO'], how='left')

# Agrupando a contagem total de votos por candidato em cada seção eleitoral nos dois turnos:
total_vote_count_grouped = df.groupby(
    ['NR_TURNO', 'NR_ZONA', 'NR_SECAO', 'NR_VOTAVEL', 'NM_VOTAVEL', 'NM_LOCAL_VOTACAO', 
     'DS_LOCAL_VOTACAO_ENDERECO', 'NR_LATITUDE', 'NR_LONGITUDE'],
     as_index=False
    )['QT_VOTOS'].sum()